# Phase 1 — TFIDF Text Representation Experiment

This notebook evaluates **TFIDF** as the text representation for Software Requirement Prioritization using **LightGBM Ranker** as the fixed baseline ranking model.

**Objective:** Determine whether SBERT produces a strong feature representation for requirement prioritization.

**Model:** `all-MiniLM-L6-v2` → 384-dimensional embeddings (pre-computed in master dataset)

**Ranker:** LightGBM (default parameters, no hyperparameter tuning)


In [9]:
import logging
import os
import json
import joblib
import warnings

import mlflow
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.preprocessing import MinMaxScaler
%pip install lightgbm
import lightgbm as lgb
from scipy.stats import spearmanr, kendalltau
from sklearn.metrics import ndcg_score

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
mlflow.set_tracking_uri("https://mlflow.smbgarasibmw.my.id")
mlflow.set_experiment("Phase_1_tfidf_Text_Representation")
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
DATASET_PATH = "dataset/Dataset_EDA_TFIDF.csv"  
OUTPUT_DIR = "outputs/phase1_tfidf"
RANDOM_STATE = 42

# TF-IDF Parameters
TFIDF_MAX_FEATURES = 5000
TFIDF_NGRAM_RANGE = (1, 2)

FEATURE_SELECTION_METHOD = "chi_square" 
K_FEATURES = 500  

TRAIN_GROUP_RATIO = 0.8 

os.makedirs(OUTPUT_DIR, exist_ok=True)
logger.info(f"Output directory initialized at: {OUTPUT_DIR}")

2026-08-08 11:40:38,316 - INFO - Output directory initialized at: outputs/phase1_tfidf


## STEP 1 — Load Dataset

Load the master dataset from `dataset/sentence_embedding.csv`. This is the source of truth containing all original metadata and pre-computed SBERT embeddings.


In [11]:
# ==========================================
# STEP 1 — Load Dataset & Validation
# ==========================================
logger.info(f"Loading dataset from {DATASET_PATH}...")
df = pd.read_csv(DATASET_PATH)
logger.info(f"Dataset shape: {df.shape}")
logger.info(f"Columns: {list(df.columns)}")

# 1. Detect requirement text column otomatis
text_keywords = ["requirement", "text", "description", "sentence", "story", "content"]
text_cols = [c for c in df.columns if any(k in c.lower() for k in text_keywords)]
if text_cols:
    logger.info(f"Detected requirement text column(s): {text_cols}")
    nama_kolom_teks = text_cols[0]  # Kunci nama kolom untuk dipanggil di sel berikutnya
else:
    raise ValueError("No text column found for TF-IDF extraction!")

# 2. Validate required columns bisnis & ranking
required_cols = {"id", "project_id", "type", "value", "effort", "risk", "stakeholder_priority", "rank"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

logger.info("All required columns present for TF-IDF ranking pipeline.")

# 3. CRITICAL SORTING: Wajib diurutkan berdasarkan project_id (sebagai query group)
df = df.sort_values(by="project_id").reset_index(drop=True)
logger.info("Dataset sorted by 'project_id' for ranking context.")

display(df.head(3))

2026-08-08 11:40:38,338 - INFO - Loading dataset from dataset/Dataset_EDA_TFIDF.csv...
2026-08-08 11:40:38,520 - INFO - Dataset shape: (698, 1437)
2026-08-08 11:40:38,520 - INFO - Columns: ['id', 'project_id', 'type', 'value', 'effort', 'risk', 'stakeholder_priority', 'priority_score', 'rank', 'cleaned_requirement_text', 'tfidf_abnormal', 'tfidf_absence', 'tfidf_accept', 'tfidf_acceptance', 'tfidf_access', 'tfidf_accessible', 'tfidf_accident', 'tfidf_accommodation', 'tfidf_accompany', 'tfidf_accord', 'tfidf_account', 'tfidf_accumulate', 'tfidf_accuracy', 'tfidf_achieve', 'tfidf_acknowledge', 'tfidf_acknowledgement', 'tfidf_across', 'tfidf_act', 'tfidf_action', 'tfidf_active', 'tfidf_activity', 'tfidf_actor', 'tfidf_actual', 'tfidf_acute', 'tfidf_ad', 'tfidf_ada', 'tfidf_adapt', 'tfidf_add', 'tfidf_addend', 'tfidf_addenda', 'tfidf_addended', 'tfidf_additional', 'tfidf_address', 'tfidf_adequate', 'tfidf_adhere', 'tfidf_adjust', 'tfidf_adjustment', 'tfidf_admin', 'tfidf_administration', '

,id,project_id,type,value,effort,risk,stakeholder_priority,priority_score,rank,cleaned_requirement_text,...,tfidf_work,tfidf_workflow,tfidf_worklist,tfidf_workshop,tfidf_workstation,tfidf_xml,tfidf_year,tfidf_yet,tfidf_zero,tfidf_zipcode
0,REQ-01,P1,FR,-0.848439,-1.015393,-1.393797,-0.8401,1.3,18,head admin finance edit delete spare part cate...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,REQ-505,P1,NFR,-0.848439,0.165847,-0.550024,-0.8401,1.4,16,cosign note record date time signature,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,REQ-506,P1,NFR,0.444597,0.165847,1.981294,-0.8401,1.2,25,record display identity addended correct note ...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## STEP 2 — Data Validation

Perform validation checks: missing values, duplicates, invalid project IDs, invalid rank values, and data types.


In [12]:
# ==========================================
# STEP 2 — Data Validation
# ==========================================
validation_report = {}

# Missing values
missing_counts = df.isnull().sum()
missing_cols = missing_counts[missing_counts > 0]
validation_report["missing_values"] = len(missing_cols)
if len(missing_cols) > 0:
    logger.warning(f"Columns with missing values:\n{missing_cols}")
else:
    logger.info("No missing values found.")

# Duplicate rows
dup_rows = df.duplicated().sum()
validation_report["duplicate_rows"] = dup_rows
if dup_rows > 0:
    logger.warning(f"Duplicate rows: {dup_rows}")
else:
    logger.info("No duplicate rows found.")

# Duplicate requirements (by id)
dup_ids = df["id"].duplicated().sum()
validation_report["duplicate_ids"] = dup_ids
if dup_ids > 0:
    logger.warning(f"Duplicate requirement IDs: {dup_ids}")
else:
    logger.info("No duplicate IDs found.")

# Invalid project_id
invalid_pid = df["project_id"].isnull().sum() + (df["project_id"].astype(str).str.strip() == "").sum()
validation_report["invalid_project_ids"] = int(invalid_pid)
if invalid_pid > 0:
    logger.warning(f"Invalid project IDs: {invalid_pid}")
else:
    logger.info("All project IDs are valid.")

# Invalid rank (should be numeric, non-negative)
invalid_rank = (~pd.to_numeric(df["rank"], errors="coerce").notna()).sum()
validation_report["invalid_rank"] = int(invalid_rank)
if invalid_rank > 0:
    logger.warning(f"Invalid rank values: {invalid_rank}")
else:
    logger.info("All rank values are valid.")

# Data types
validation_report["dtypes"] = {c: str(dt) for c, dt in df.dtypes.items()}

logger.info("=== Validation Report ===")
for k, v in validation_report.items():
    logger.info(f"  {k}: {v}")

2026-08-08 11:40:38,592 - INFO - No missing values found.
2026-08-08 11:40:38,759 - INFO - No duplicate rows found.
2026-08-08 11:40:38,766 - INFO - No duplicate IDs found.
2026-08-08 11:40:38,766 - INFO - All project IDs are valid.
2026-08-08 11:40:38,772 - INFO - All rank values are valid.
2026-08-08 11:40:38,782 - INFO - === Validation Report ===
2026-08-08 11:40:38,782 - INFO -   missing_values: 0
2026-08-08 11:40:38,782 - INFO -   duplicate_rows: 0
2026-08-08 11:40:38,788 - INFO -   duplicate_ids: 0
2026-08-08 11:40:38,788 - INFO -   invalid_project_ids: 0
2026-08-08 11:40:38,789 - INFO -   invalid_rank: 0
2026-08-08 11:40:38,789 - INFO -   dtypes: {'id': 'object', 'project_id': 'object', 'type': 'object', 'value': 'float64', 'effort': 'float64', 'risk': 'float64', 'stakeholder_priority': 'float64', 'priority_score': 'float64', 'rank': 'int64', 'cleaned_requirement_text': 'object', 'tfidf_abnormal': 'float64', 'tfidf_absence': 'float64', 'tfidf_accept': 'float64', 'tfidf_acceptanc

## STEP 3 — Text Representation (TF-IDF) & MLflow Run Start



In [13]:
logger.info("Initializing MLflow run for pre-computed TF-IDF dataset...")
run = mlflow.start_run(run_name="PreComputed_TFIDF_Experiment")

# Filter semua kolom yang namanya diawali dengan 'tfidf_'
tfidf_cols = [c for c in df.columns if c.startswith("tfidf_")]
X_tfidf = df[tfidf_cols].values  # Langsung jadi matriks fitur teks

mlflow.log_param("text_representation", "Pre-Computed TF-IDF")
mlflow.log_param("tfidf_features_count", X_tfidf.shape[1])
logger.info(f"Loaded pre-computed TF-IDF matrix. Shape: {X_tfidf.shape}")

2026-08-08 11:40:38,808 - INFO - Initializing MLflow run for pre-computed TF-IDF dataset...
2026-08-08 11:40:39,604 - INFO - Loaded pre-computed TF-IDF matrix. Shape: (698, 1427)


## STEP 4 - Feature Selection

In [14]:
logger.info(f"Starting feature selection using method: {FEATURE_SELECTION_METHOD}...")

# 1. Tarik target y (kolom 'rank' dari dataset lu)
y = df["rank"].values

# 2. Log parameter ke MLflow
mlflow.log_param("feature_selection_method", FEATURE_SELECTION_METHOD)
mlflow.log_param("selected_k_features", K_FEATURES)

# 3. Kondisional: Agent lu tinggal ganti parameter di config paling atas
if FEATURE_SELECTION_METHOD == "chi_square":
    selector = SelectKBest(score_func=chi2, k=K_FEATURES)
elif FEATURE_SELECTION_METHOD == "mutual_information":
    # Menghitung informasi timbal-balik antara kata dengan label rank
    selector = SelectKBest(score_func=mutual_info_classif, k=K_FEATURES)
else:
    raise ValueError(f"Metode '{FEATURE_SELECTION_METHOD}' tidak dikenal!")

# 4. Transformasi data untuk mengambil K fitur teks terbaik
X_text_selected = selector.fit_transform(X_tfidf, y)

logger.info(f"Feature selection completed. Reduced matrix shape: {X_text_selected.shape}")

2026-08-08 11:40:39,622 - INFO - Starting feature selection using method: chi_square...
2026-08-08 11:40:39,839 - INFO - Feature selection completed. Reduced matrix shape: (698, 500)


## STEP 5 — Feature Fusion


In [15]:
num_features = ["value", "effort", "risk", "stakeholder_priority"]
logger.info(f"Numerical business features to fuse: {num_features}")

# 1. Ambil nilai numerik dari DataFrame menjadi NumPy Array
X_num = df[num_features].values

# 2. Pastikan X_text_selected dalam bentuk dense array (bukan sparse matrix) sebelum digabung
X_text_dense = X_text_selected.toarray() if hasattr(X_text_selected, "toarray") else X_text_selected

# 3. FUSI DATA: Gabungkan fitur numerik bisnis dan fitur teks terpilih ke samping (horizontal)
X_fused = np.hstack((X_num, X_text_dense))

# 4. Catat dimensi matriks final ke MLflow
mlflow.log_param("total_fused_features", X_fused.shape[1])

logger.info(f"Fusion completed:")
logger.info(f" - Numerical dimension: {X_num.shape[1]}")
logger.info(f" - Selected text dimension: {X_text_dense.shape[1]}")
logger.info(f" - Final Fused Feature Matrix shape (X): {X_fused.shape}")

# Mengintip beberapa baris pertama hasil fusi (fitur numerik bisnis berada di 4 kolom terdepan)
print("\nFirst 3 rows of final feature matrix (X) looks like:")
print(X_fused[:3, :8]) # Intip 4 fitur numerik + 4 fitur teks pertama

2026-08-08 11:40:39,863 - INFO - Numerical business features to fuse: ['value', 'effort', 'risk', 'stakeholder_priority']
2026-08-08 11:40:39,955 - INFO - Fusion completed:
2026-08-08 11:40:39,957 - INFO -  - Numerical dimension: 4
2026-08-08 11:40:39,957 - INFO -  - Selected text dimension: 500
2026-08-08 11:40:39,959 - INFO -  - Final Fused Feature Matrix shape (X): (698, 504)



First 3 rows of final feature matrix (X) looks like:
[[-0.84843916 -1.01539269 -1.39379651 -0.84009961  0.          0.
   0.          0.        ]
 [-0.84843916  0.16584747 -0.55002378 -0.84009961  0.          0.
   0.          0.        ]
 [ 0.44459694  0.16584747  1.98129444 -0.84009961  0.          0.
   0.          0.        ]]


## STEP 6 — Feature Encoding

One-hot encode the `type` column (FR/NFR) and append to the feature matrix.


In [16]:
logger.info("Performing One-Hot Encoding on 'type' column...")

# 1. Lakukan One-Hot Encoding pada kolom 'type' dan langsung ambil nilai NumPy-nya (.values)
encoded_type_df = pd.get_dummies(df["type"], prefix="type")
X_encoded_type = encoded_type_df.values

logger.info(f"One-hot encoded type columns: {list(encoded_type_df.columns)}")

# 2. FUSI DATA LAGI: Tempelkan hasil encoding secara horizontal ke matriks X_fused kita
X_fused = np.hstack((X_fused, X_encoded_type))

# 3. Log parameter dimensi terbaru ke MLflow
mlflow.log_param("features_count_after_encoding", X_fused.shape[1])

logger.info(f"Feature encoding completed.")
logger.info(f" - Encoded type dimension: {X_encoded_type.shape[1]}")
logger.info(f" - Final Feature Matrix shape (X) after encoding: {X_fused.shape}")

2026-08-08 11:40:39,977 - INFO - Performing One-Hot Encoding on 'type' column...
2026-08-08 11:40:39,979 - INFO - One-hot encoded type columns: ['type_FR', 'type_NFR']
2026-08-08 11:40:40,065 - INFO - Feature encoding completed.
2026-08-08 11:40:40,068 - INFO -  - Encoded type dimension: 2
2026-08-08 11:40:40,068 - INFO -  - Final Feature Matrix shape (X) after encoding: (698, 506)


## STEP 7 — Query Group Construction

Prepare the dataset for Learning to Rank:
- **Query/group identifier:** `project_id`
- **Ranking label:** `rank`
- Verify every project contains multiple requirements.


In [17]:
logger.info("Transforming ranks into LambdaRank-compatible labels...")

# 1. Transformasi rank ke skor relevansi (0 = terburuk, N-1 = terbaik)
df["label"] = df.groupby("project_id")["rank"].transform(
    lambda x: x.rank(method="dense", ascending=False).astype(int) - 1
)

# 2. Definisikan nilai target y akhir dari kolom label baru
y = df["label"].values
query_ids = df["project_id"]

# 3. Hitung distribusi ukuran tiap kelompok proyek (Query Group)
group_sizes = query_ids.value_counts()
num_groups = len(group_sizes)

logger.info(f"Number of query groups (projects): {num_groups}")
logger.info(f"Group size stats:\n{group_sizes.describe()}")

# Peringatan untuk proyek yang isinya cuma 1 kebutuhan
single_req_groups = (group_sizes == 1).sum()
if single_req_groups > 0:
    logger.warning(f"Groups with only 1 requirement: {single_req_groups} — these cannot be ranked")

# 4. Kunci ukuran grup untuk kebutuhan parameter training LightGBM Ranker
group_counts = group_sizes.sort_index().values  # Urut berdasarkan project_id
logger.info(f"Group sizes array for LightGBM (first 10): {group_counts[:10]}...")

# 5. Log metrik distribusi grup dan label ke MLflow
mlflow.log_param("num_query_groups", num_groups)
mlflow.log_param("label_min_value", int(y.min()))
mlflow.log_param("label_max_value", int(y.max()))

logger.info(f"Label validation -> Range: [{y.min()}, {y.max()}], Unique labels: {sorted(np.unique(y))[:20]}...")

2026-08-08 11:40:40,081 - INFO - Transforming ranks into LambdaRank-compatible labels...
2026-08-08 11:40:40,103 - INFO - Number of query groups (projects): 15
2026-08-08 11:40:40,108 - INFO - Group size stats:
count     15.000000
mean      46.533333
std       64.790725
min        3.000000
25%       10.000000
50%       18.000000
75%       49.500000
max      231.000000
Name: count, dtype: float64
2026-08-08 11:40:40,108 - INFO - Group sizes array for LightGBM (first 10): [ 47   8  18  10  19   9   3 231  99 146]...
2026-08-08 11:40:40,363 - INFO - Label validation -> Range: [0, 20], Unique labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19)]...


## STEP 7 — Train/Test Split

Use `GroupShuffleSplit` to ensure the same project never appears in both train and test sets. Fixed `random_state=42` for reproducibility.


In [20]:
from sklearn.model_selection import GroupShuffleSplit

TEST_SIZE = 0.2       # Contoh: 20% data untuk test set
RANDOM_STATE = 42     # Seed untuk reproducibility

gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)
y_train = y[train_idx]
y_test = y[test_idx]
groups_train = groups[train_idx]
groups_test = groups[test_idx]

# Recompute group counts for train/test
train_group_counts = pd.Series(groups_train).value_counts().sort_index().values
test_group_counts = pd.Series(groups_test).value_counts().sort_index().values

logger.info(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
logger.info(f"Train groups: {len(np.unique(groups_train))}, Test groups: {len(np.unique(groups_test))}")

# Verify no overlap
train_projects = set(np.unique(groups_train))
test_projects = set(np.unique(groups_test))
overlap = train_projects & test_projects
if overlap:
    raise ValueError(f"Project overlap between train and test: {overlap}")
logger.info("No project overlap between train and test sets. Split is clean.")


NameError: name 'X' is not defined

## STEP 8 — LightGBM Ranker Training

Train a baseline LightGBM Ranker with reasonable default parameters. No hyperparameter tuning — the goal is to evaluate SBERT representation quality, not to optimize the ranker.


In [ ]:
max_group_size = df.groupby("project_id").size().max()
num_leaves = min(max_group_size, 255)

ranker = lgb.LGBMRanker(
    objective="lambdarank",
    boosting_type="gbdt",
    n_estimators=100,
    num_leaves=num_leaves,
    learning_rate=0.1,
    min_child_samples=10,
    random_state=RANDOM_STATE,
    verbose=-1
)

logger.info("Training LightGBM Ranker...")
ranker.fit(
    X_train, y_train,
    group=train_group_counts,
    eval_set=[(X_test, y_test)],
    eval_group=[test_group_counts],
    eval_metric=["ndcg"],
    callbacks=[lgb.log_evaluation(0)]
)
logger.info("Training complete.")

logger.info(f"Feature importances (top 10):\n"
            f"{pd.Series(ranker.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)}")


2026-07-19 21:55:52,229 - INFO - Training LightGBM Ranker...
2026-07-19 21:55:54,278 - INFO - Training complete.
2026-07-19 21:55:54,279 - INFO - Feature importances (top 10):
effort           96
value            66
embedding_94     47
embedding_233    41
embedding_106    40
embedding_120    39
embedding_43     34
embedding_169    34
embedding_231    33
embedding_70     32
dtype: int32


## STEP 9 — Model Evaluation

Evaluate using:
- **NDCG@5, NDCG@10** — Normalized Discounted Cumulative Gain
- **MAP** — Mean Average Precision
- **Spearman Rank Correlation**
- **Kendall Tau**


In [ ]:
y_pred = ranker.predict(X_test)

# Compute NDCG per query group, then average
test_project_ids = groups_test
ndcg5_scores = []
ndcg10_scores = []
map_per_group = []

for pid in np.unique(test_project_ids):
    mask = test_project_ids == pid
    y_true_group = y_test[mask]
    y_pred_group = y_pred[mask]
    n = len(y_true_group)

    # NDCG
    if n >= 5:
        k5 = min(5, n)
        ndcg5_scores.append(ndcg_score(y_true_group.reshape(1, -1),
                                        y_pred_group.reshape(1, -1), k=k5))
    if n >= 10:
        k10 = min(10, n)
        ndcg10_scores.append(ndcg_score(y_true_group.reshape(1, -1),
                                         y_pred_group.reshape(1, -1), k=k10))

    # MAP (binarize: top half of labels = relevant)
    if len(np.unique(y_true_group)) > 1:
        threshold = y_true_group.max() * 0.5
        y_bin = (y_true_group >= threshold).astype(int)
        if y_bin.sum() > 0 and y_bin.sum() < len(y_bin):
            map_per_group.append(average_precision_score(y_bin, y_pred_group))

ndcg5 = float(np.mean(ndcg5_scores)) if ndcg5_scores else 0.0
ndcg10 = float(np.mean(ndcg10_scores)) if ndcg10_scores else 0.0
map_score = float(np.mean(map_per_group)) if map_per_group else 0.0

spearman_corr, spearman_p = spearmanr(y_test, y_pred)
kendall_corr, kendall_p = kendalltau(y_test, y_pred)

metrics = {
    "NDCG_at_5": round(float(ndcg5), 6),
    "NDCG_at_10": round(float(ndcg10), 6),
    "MAP": round(float(map_score), 6),
    "Spearman": round(float(spearman_corr), 6),
    "Spearman_pvalue": round(float(spearman_p), 6),
    "KendallTau": round(float(kendall_corr), 6),
    "KendallTau_pvalue": round(float(kendall_p), 6)
}

logger.info("=== Evaluation Metrics ===")
for k, v in metrics.items():
    logger.info(f"  {k}: {v}")

display(pd.DataFrame([metrics]))


2026-07-19 21:55:54,307 - INFO - === Evaluation Metrics ===
2026-07-19 21:55:54,309 - INFO -   NDCG_at_5: 0.687576
2026-07-19 21:55:54,310 - INFO -   NDCG_at_10: 0.751521
2026-07-19 21:55:54,310 - INFO -   MAP: 0.751672
2026-07-19 21:55:54,311 - INFO -   Spearman: 0.518793
2026-07-19 21:55:54,311 - INFO -   Spearman_pvalue: 0.0
2026-07-19 21:55:54,312 - INFO -   KendallTau: 0.370093
2026-07-19 21:55:54,313 - INFO -   KendallTau_pvalue: 0.0


,NDCG_at_5,NDCG_at_10,MAP,Spearman,Spearman_pvalue,KendallTau,KendallTau_pvalue
0,0.687576,0.751521,0.751672,0.518793,0.0,0.370093,0.0


## STEP 10 — Save Outputs

Save all experiment outputs to `outputs/phase1_sbert/`:
- Trained LightGBM model
- Generated SBERT dataset (for Phase 2 input)
- Evaluation metrics (JSON)
- Predictions
- Feature list


In [ ]:
# 1. Save trained model
model_path = os.path.join(OUTPUT_DIR, "lgbm_ranker.pkl")
joblib.dump(ranker, model_path)
logger.info(f"Model saved: {model_path}")

# 2. Save generated dataset (Phase 2 input)
dataset_out = X.copy()
dataset_out["label"] = y
dataset_out["rank"] = df["rank"].values
dataset_out["project_id"] = groups
dataset_path = os.path.join(OUTPUT_DIR, "sbert_dataset.csv")
dataset_out.to_csv(dataset_path, index=False)
logger.info(f"Generated dataset saved: {dataset_path} (shape: {dataset_out.shape})")

# 3. Save evaluation metrics
metrics_path = os.path.join(OUTPUT_DIR, "evaluation_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
logger.info(f"Metrics saved: {metrics_path}")

# 4. Save predictions
predictions_df = pd.DataFrame({
    "project_id": groups_test,
    "true_label": y_test,
    "predicted_score": y_pred
})
pred_path = os.path.join(OUTPUT_DIR, "predictions.csv")
predictions_df.to_csv(pred_path, index=False)
logger.info(f"Predictions saved: {pred_path}")

# 5. Save feature list
features_path = os.path.join(OUTPUT_DIR, "feature_list.txt")
with open(features_path, "w") as f:
    f.write("\n".join(X.columns.tolist()))
logger.info(f"Feature list saved: {features_path}")


2026-07-19 21:55:54,344 - INFO - Model saved: outputs/phase1_sbert\lgbm_ranker.pkl
2026-07-19 21:55:54,500 - INFO - Generated dataset saved: outputs/phase1_sbert\sbert_dataset.csv (shape: (698, 256))
2026-07-19 21:55:54,500 - INFO - Metrics saved: outputs/phase1_sbert\evaluation_metrics.json
2026-07-19 21:55:54,504 - INFO - Predictions saved: outputs/phase1_sbert\predictions.csv
2026-07-19 21:55:54,505 - INFO - Feature list saved: outputs/phase1_sbert\feature_list.txt


## STEP 11 — MLflow Logging

Track the experiment using MLflow: log parameters, metrics, and artifacts.


In [ ]:
with mlflow.start_run(run_name="SBERT_all-MiniLM-L6-v2_LightGBM") as run:
    # Log parameters
    mlflow.log_param("embedding_model", EMBEDDING_MODEL_NAME)
    mlflow.log_param("embedding_dimension", actual_dim)
    mlflow.log_param("train_size", len(X_train))
    mlflow.log_param("test_size", len(X_test))
    mlflow.log_param("num_train_groups", len(np.unique(groups_train)))
    mlflow.log_param("num_test_groups", len(np.unique(groups_test)))
    mlflow.log_param("random_state", RANDOM_STATE)
    mlflow.log_param("test_size_ratio", TEST_SIZE)
    mlflow.log_param("num_features", X.shape[1])
    mlflow.log_param("ranker_objective", "lambdarank")
    mlflow.log_param("ranker_n_estimators", 100)
    mlflow.log_param("ranker_num_leaves", num_leaves)

    # Log metrics
    mlflow.log_metrics(metrics)

    # Log artifacts
    mlflow.log_artifact(model_path, artifact_path="model")
    mlflow.log_artifact(metrics_path, artifact_path="metrics")
    mlflow.log_artifact(features_path, artifact_path="features")
    mlflow.log_artifact(dataset_path, artifact_path="dataset")
    mlflow.log_artifact(pred_path, artifact_path="predictions")

    logger.info(f"MLflow run ID: {run.info.run_id}")
    logger.info(f"MLflow experiment logged successfully.")


2026-07-19 21:55:55,015 - INFO - MLflow run ID: 1b5084324d1443f98448ff81967481a3
2026-07-19 21:55:55,016 - INFO - MLflow experiment logged successfully.


## Experiment Summary

### What was done
1. **Dataset:** Loaded master dataset with pre-computed SBERT embeddings
2. **Features:** 4 numerical features + embedding dimensions + one-hot encoded type
3. **Model:** LightGBM Ranker (Lambdarank objective, default parameters)
4. **Split:** GroupShuffleSplit (30% test, no project overlap)
5. **Evaluation:** NDCG@5, NDCG@10, MAP, Spearman, Kendall Tau


In [ ]:
summary_data = {
    "Metric": ["NDCG@5", "NDCG@10", "MAP", "Spearman rho", "Kendall tau"],
    "Value": [
        metrics["NDCG_at_5"],
        metrics["NDCG_at_10"],
        metrics["MAP"],
        metrics["Spearman"],
        metrics["KendallTau"]
    ]
}
summary_df = pd.DataFrame(summary_data)

logger.info("=== Experiment Summary ===")
logger.info(f"Embedding model: {EMBEDDING_MODEL_NAME}")
logger.info(f"Embedding dimension: {actual_dim}")
logger.info(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
logger.info(f"Train groups: {len(np.unique(groups_train))}, Test groups: {len(np.unique(groups_test))}")
logger.info(f"Total features: {X.shape[1]}")
logger.info(f"\n{summary_df.to_string(index=False)}")

display(summary_df)


2026-07-19 21:55:55,042 - INFO - === Experiment Summary ===
2026-07-19 21:55:55,043 - INFO - Embedding model: all-MiniLM-L6-v2
2026-07-19 21:55:55,045 - INFO - Embedding dimension: 247
2026-07-19 21:55:55,046 - INFO - Train size: 434, Test size: 264
2026-07-19 21:55:55,047 - INFO - Train groups: 10, Test groups: 5
2026-07-19 21:55:55,048 - INFO - Total features: 253
2026-07-19 21:55:55,050 - INFO - 
      Metric    Value
      NDCG@5 0.687576
     NDCG@10 0.751521
         MAP 0.751672
Spearman rho 0.518793
 Kendall tau 0.370093


,Metric,Value
0,NDCG@5,0.687576
1,NDCG@10,0.751521
2,MAP,0.751672
3,Spearman rho,0.518793
4,Kendall tau,0.370093


### Outputs
All artifacts saved to `outputs/phase1_sbert/`. The generated dataset is ready for Phase 2 (Ranking Algorithm Experiment).

---
*Phase 1 — SBERT Text Representation Experiment complete.*